# Train, compare, and save the winning model

This notebook does **all** the model training for the deployed app. Three things happen here, and **none** of them happen at runtime in `app.py`:

1. Fit three candidate algorithms — Logistic Regression, K-Nearest Neighbors, Decision Tree — on the same train / test split.
2. Score each on the held-out test set; persist a comparison table to `model_comparison.csv`.
3. Pick the F1 winner; persist *only* that fitted `Pipeline` to `model.joblib`.

`app.py` then loads the artifact, renders the comparison table as a static Model Card, and serves predictions. **It never calls `.fit()`.** That separation is the whole point of the assignment.

> **For your assignment:** keep this exact shape — multiple candidates trained offline, one winner persisted, comparison rendered statically in the app. Replace the dataset / target / candidate algorithms with whichever ones match your final-assignment work.

## 1. Imports

In [ ]:
import time
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

## 2. Load the bundled sample

5,000-row stratified sample of the heart-disease CSV from `lecture_07_pandas/reading_material/`. The full file (31 MB / 630k rows) is too heavy for a Hugging Face Space; the sampling code is in section 8 below for reproducibility.

In [ ]:
HERE = Path.cwd()
df = pd.read_csv(HERE / "heart_disease_sample.csv")
print(df.shape)
df.head()

## 3. One train / test split, used by every candidate

Every algorithm we benchmark below is fit on the **same** train split and scored on the **same** test split, with the **same** random seed. That's what makes the comparison fair.

In [ ]:
TARGET = "Heart Disease"
y = (df[TARGET] == "Presence").astype(int)
X = df.drop(columns=[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)
print("train", X_train.shape, "test", X_test.shape)

## 4. Define the three candidate Pipelines

Each candidate is a `Pipeline` so that the fitted scaler travels with the classifier when we save it. Hyperparameters are reasonable defaults — the point of this assignment is the *deployment pattern*, not hyperparameter optimisation.

In [ ]:
candidates = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(C=1.0, max_iter=1000, random_state=0)),
    ]),
    "K-Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    KNeighborsClassifier(n_neighbors=5)),
    ]),
    "Decision Tree": Pipeline([
        ("clf",    DecisionTreeClassifier(max_depth=5, random_state=0)),
    ]),
}

hyperparam_summary = {
    "Logistic Regression": "C=1.0",
    "K-Nearest Neighbors": "n_neighbors=5",
    "Decision Tree":       "max_depth=5",
}

## 5. Fit each candidate, time it, score it

In [ ]:
rows = []
fitted = {}

for name, pipe in candidates.items():
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    y_pred = pipe.predict(X_test)
    rows.append({
        "Algorithm":       name,
        "Hyperparameter":  hyperparam_summary[name],
        "Accuracy":        round(accuracy_score(y_test, y_pred), 4),
        "F1":              round(f1_score(y_test, y_pred), 4),
        "Fit time (s)":    round(fit_time, 3),
    })
    fitted[name] = pipe

comparison = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
comparison

## 6. Persist artifacts

We persist **two** files for the app to read at startup:

- `model.joblib` — a single dict bundling **all three fitted Pipelines** (LogReg, KNN, Tree), plus metadata: feature names, class labels, the F1 winner's name, and the short justification rendered on the Model Card.
- `model_comparison.csv` — the test-set scores for every candidate. The Model Card tab renders this as a static table.

Why ship all three pipelines, not just the winner? Because the Predict tab lets the user pick which algorithm to inference against. That makes the Model Card claim *verifiable* — a recruiter can run a borderline patient through all three and see which one disagrees. Crucially, **the app still does no training**: it just loads three pre-fitted artifacts and serves them.


In [ ]:
winner_name = comparison.iloc[0]["Algorithm"]
print(f"Winner: {winner_name} (F1 = {comparison.iloc[0]['F1']})")

WINNER_JUSTIFICATION = (
    f"{winner_name} wins on F1 against KNN and a depth-5 Decision Tree on the same "
    "train/test split. Logistic Regression handles the (mostly numeric, weakly correlated) "
    "heart-disease features well, and its scaled, regularised linear boundary generalises "
    "better than KNN's local averaging or the tree's hard splits at this sample size."
)

joblib.dump({
    "pipelines":         fitted,           # dict: {algo_name: fitted Pipeline} for all three candidates
    "feature_names":     list(X.columns),
    "target_name":       TARGET,
    "positive_label":    "Presence",
    "negative_label":    "Absence",
    "winner_name":       winner_name,      # used as the Predict tab's default algorithm
    "justification":     WINNER_JUSTIFICATION,
}, HERE / "model.joblib")

comparison.to_csv(HERE / "model_comparison.csv", index=False)
print("saved model.joblib (3 pipelines + metadata) and model_comparison.csv")

## 7. Sanity check — reload, predict, classification report

Always reload in a fresh context (in real life, a fresh Python process — here, a fresh cell will do) and verify prediction still works. If this cell fails, deployment will fail too.

In [ ]:
reloaded = joblib.load(HERE / "model.joblib")
print("pipelines bundled:", list(reloaded["pipelines"]))
print("default (winner) :", reloaded["winner_name"])

one_row = X_test.iloc[[0]]
for name, pipe in reloaded["pipelines"].items():
    pred = pipe.predict(one_row)[0]
    proba = pipe.predict_proba(one_row).round(3)
    print(f"  {name:22s} -> pred={pred}, proba={proba.tolist()[0]}")

winner_pipe = reloaded["pipelines"][reloaded["winner_name"]]
print()
print(classification_report(
    y_test, winner_pipe.predict(X_test),
    target_names=["Absence", "Presence"],
))

## 8. (Optional) How the bundled sample was created

The cell below regenerates `heart_disease_sample.csv` from the full lecture CSV. You only need to run this if you want to refresh the sample — the file is already in the folder.

In [ ]:
# from sklearn.model_selection import train_test_split
#
# FULL = Path("../../lectures_07_13_pandas_plots_scikit/lecture_07_pandas/reading_material/predict_heart_disease_train.csv")
# full_df = pd.read_csv(FULL)
# sample, _ = train_test_split(full_df, train_size=5000, stratify=full_df["Heart Disease"], random_state=0)
# sample = sample.drop(columns=["id"]).reset_index(drop=True)
# sample.to_csv(HERE / "heart_disease_sample.csv", index=False)